# Notebook 14 — Manuscript Drafting Package

**Project:** Machine learning prediction of rapid motor progression in Parkinson’s disease using PPMI data  
**Purpose:** Create a manuscript-ready writing package from the approved outputs of Notebooks 01–13.

This notebook does **not** train new models and does **not** re-tune model parameters. It consolidates the approved results into a structured drafting package for manuscript preparation.

## Objective

Generate a manuscript drafting package including:

1. Final title options and key messages.
2. Structured abstract.
3. Methods draft.
4. Results draft.
5. Table captions and figure legends.
6. Limitations and future work.
7. Data availability and PPMI acknowledgement text.
8. Reporting checklist and submission readiness checklist.

## Scientific Background

Parkinson’s disease progression is heterogeneous. The approved analysis evaluates whether baseline clinical variables, DaTSCAN/SBR imaging-derived measures, and SAA biomarker features can predict rapid motor progression using PPMI data.

The final interpretation from Notebook 13 is that the study should be reported as a **reproducible machine learning benchmark**, not as a clinically deployable high-performance prediction model.

## Dataset Verification

This notebook uses only approved summary outputs from prior notebooks. It does not use raw participant-level data directly unless required to verify row counts already generated in previous steps.

Required Notebook 13 outputs:

- `01_table1_cohort_and_outcome_summary.csv`
- `03_feature_set_summary.csv`
- `04_manuscript_ready_model_performance_table.csv`
- `05_incremental_value_summary.csv`
- `06_final_model_selection_decision_table.csv`
- `07_scientific_interpretation_table.csv`
- `08_quality_control_checklist.csv`

The notebook searches the project output folders automatically.

## Code

In [ ]:
# ============================================================
# 01. Mount Google Drive and define project paths
# ============================================================

from pathlib import Path
import os
import json
import textwrap
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Google Colab or Google Drive mount unavailable.")

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression") if IN_COLAB else Path.cwd()

OUT_DIR = PROJECT_DIR / "outputs" / "notebook_14_manuscript_drafting_package"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR:", OUT_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())

In [ ]:
# ============================================================
# 02. Locate required Notebook 13 files
# ============================================================

REQUIRED_FILES = {
    "table1": "01_table1_cohort_and_outcome_summary.csv",
    "feature_summary": "03_feature_set_summary.csv",
    "performance": "04_manuscript_ready_model_performance_table.csv",
    "incremental": "05_incremental_value_summary.csv",
    "decision": "06_final_model_selection_decision_table.csv",
    "interpretation": "07_scientific_interpretation_table.csv",
    "qc13": "08_quality_control_checklist.csv"
}

SEARCH_ROOTS = [
    PROJECT_DIR / "outputs",
    PROJECT_DIR,
    Path("/mnt/data")  # useful only when run in this environment
]

def find_file(filename, search_roots=SEARCH_ROOTS):
    hits = []
    for root in search_roots:
        if root.exists():
            hits.extend(list(root.rglob(filename)))
    hits = sorted(set(hits), key=lambda p: len(str(p)))
    return hits[0] if hits else None

found_files = {}
missing = []

for key, filename in REQUIRED_FILES.items():
    path = find_file(filename)
    found_files[key] = path
    if path is None:
        missing.append(filename)

print("Located files:")
for key, path in found_files.items():
    print(f"- {key}: {path}")

if missing:
    raise FileNotFoundError(
        "Missing required Notebook 13 outputs:\n"
        + "\n".join(missing)
        + "\n\nPlease place Notebook 13 output files under PROJECT_DIR/outputs/."
    )

In [ ]:
# ============================================================
# 03. Load approved summary outputs
# ============================================================

table1 = pd.read_csv(found_files["table1"])
feature_summary = pd.read_csv(found_files["feature_summary"])
performance = pd.read_csv(found_files["performance"])
incremental = pd.read_csv(found_files["incremental"])
decision = pd.read_csv(found_files["decision"])
interpretation = pd.read_csv(found_files["interpretation"])
qc13 = pd.read_csv(found_files["qc13"])

print("table1:", table1.shape)
print("feature_summary:", feature_summary.shape)
print("performance:", performance.shape)
print("incremental:", incremental.shape)
print("decision:", decision.shape)
print("interpretation:", interpretation.shape)
print("qc13:", qc13.shape)

display(table1.head())
display(performance.head())

In [ ]:
# ============================================================
# 04. Helper functions for safe value extraction
# ============================================================

def get_table1_value(metric_contains, default="Not available"):
    mask = table1["metric"].astype(str).str.contains(metric_contains, case=False, na=False)
    if mask.any():
        return table1.loc[mask, "value"].iloc[0]
    return default

def fmt_num(x, digits=3, default="NA"):
    try:
        if pd.isna(x):
            return default
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)

def best_by_metric(df, metric):
    if metric not in df.columns:
        return None
    temp = df.copy()
    temp[metric] = pd.to_numeric(temp[metric], errors="coerce")
    temp = temp.dropna(subset=[metric])
    if temp.empty:
        return None
    return temp.sort_values(metric, ascending=False).iloc[0]

def md_table_from_df(df, max_rows=20):
    if df is None or df.empty:
        return "_No data available._"
    d = df.head(max_rows).copy()
    return d.to_markdown(index=False)

n_total = get_table1_value("Participants")
n_rapid = get_table1_value("Rapid progressors, n")
pct_rapid = get_table1_value("Rapid progressors, %")
baseline_np3_mean = get_table1_value("baseline_NP3TOT, mean")
baseline_np3_median = get_table1_value("baseline_NP3TOT, median")

best_roc = best_by_metric(performance, "roc_auc")
best_pr = best_by_metric(performance, "pr_auc")
best_bal = best_by_metric(performance, "balanced_accuracy")
best_f1 = best_by_metric(performance, "f1")

summary_values = {
    "n_total": n_total,
    "n_rapid": n_rapid,
    "pct_rapid": pct_rapid,
    "baseline_np3_mean": baseline_np3_mean,
    "baseline_np3_median": baseline_np3_median,
    "best_roc_auc": fmt_num(best_roc["roc_auc"]) if best_roc is not None else "NA",
    "best_roc_model": str(best_roc["model"]) if best_roc is not None else "NA",
    "best_roc_feature_set": str(best_roc.get("feature_set", "NA")) if best_roc is not None else "NA",
    "best_pr_auc": fmt_num(best_pr["pr_auc"]) if best_pr is not None else "NA",
    "best_pr_model": str(best_pr["model"]) if best_pr is not None else "NA",
    "best_balanced_accuracy": fmt_num(best_bal["balanced_accuracy"]) if best_bal is not None else "NA",
    "best_f1": fmt_num(best_f1["f1"]) if best_f1 is not None else "NA",
}

summary_values

In [ ]:
# ============================================================
# 05. Create title options and key messages
# ============================================================

title_options = [
    "A Reproducible Machine Learning Benchmark for Predicting Rapid Motor Progression in Parkinson’s Disease Using PPMI Data",
    "Clinical, DaTSCAN, and SAA Biomarker Feature Sets for Predicting Parkinson’s Disease Motor Progression: A Reproducible Machine Learning Benchmark",
    "Predicting Rapid Motor Progression in Parkinson’s Disease Using Reproducible Machine Learning: Evidence from PPMI Clinical, Imaging, and Biomarker Data"
]

key_messages = [
    "This study should be positioned as a reproducible benchmark rather than a clinically deployable prediction model.",
    f"The final analytic cohort included {summary_values['n_total']} participants with Parkinson’s disease, including {summary_values['n_rapid']} rapid progressors ({summary_values['pct_rapid']}%).",
    f"The best held-out ROC-AUC was {summary_values['best_roc_auc']}, indicating modest discrimination.",
    "Clinical-only models should be retained as the primary baseline comparator.",
    "DaTSCAN/SBR and SAA biomarker features should be reported as secondary or exploratory analyses because incremental gains were limited or inconsistent.",
    "The manuscript should transparently emphasize limitations, class imbalance, internal validation only, and the need for external validation."
]

title_md = "# Title options and key messages\n\n"
title_md += "## Recommended title\n\n"
title_md += f"**{title_options[0]}**\n\n"
title_md += "## Alternative titles\n\n"
for i, title in enumerate(title_options[1:], start=2):
    title_md += f"{i}. {title}\n"
title_md += "\n## Key messages\n\n"
for msg in key_messages:
    title_md += f"- {msg}\n"

(OUT_DIR / "01_title_and_key_messages.md").write_text(title_md, encoding="utf-8")
print(title_md)

In [ ]:
# ============================================================
# 06. Create structured abstract draft
# ============================================================

abstract = f'''
# Structured Abstract Draft

## Background
Motor progression in Parkinson’s disease is heterogeneous, and reliable early prediction remains challenging. Publicly available longitudinal cohorts such as the Parkinson’s Progression Markers Initiative (PPMI) provide an opportunity to evaluate reproducible machine learning workflows using clinical, imaging-derived, and biomarker features.

## Objective
To develop and evaluate a reproducible machine learning benchmark for predicting rapid motor progression in Parkinson’s disease using baseline clinical variables, DaTSCAN/SBR imaging-derived measures, and SAA biomarker features from PPMI.

## Methods
Participants with Parkinson’s disease were selected from PPMI. The primary outcome was rapid motor progression, defined from annualized change in MDS-UPDRS Part III from baseline to the selected follow-up visit. Baseline clinical predictors were evaluated first, followed by secondary models incorporating DaTSCAN/SBR features and exploratory models incorporating SAA biomarker features. Preprocessing, imputation, scaling, and encoding were performed using training data only and applied to held-out test data to reduce data leakage. Models were evaluated using ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, and F1-score.

## Results
The analytic cohort included {summary_values['n_total']} participants, of whom {summary_values['n_rapid']} ({summary_values['pct_rapid']}%) were classified as rapid progressors. Across clinical, DaTSCAN/SBR, and SAA-augmented models, held-out discrimination was modest. The best ROC-AUC was {summary_values['best_roc_auc']} using {summary_values['best_roc_model']} with the {summary_values['best_roc_feature_set']} feature set. Incremental improvements from DaTSCAN/SBR and SAA biomarkers were limited or inconsistent across performance metrics.

## Conclusions
In this reproducible PPMI benchmark, prediction of rapid motor progression remained challenging. Baseline clinical, DaTSCAN/SBR, and SAA biomarker features produced only modest discrimination, and the resulting models should not be interpreted as clinically deployable. The workflow provides a transparent benchmark for future multimodal prediction studies and highlights the need for external validation and stronger longitudinal predictors.
'''.strip()

(OUT_DIR / "02_structured_abstract_draft.md").write_text(abstract, encoding="utf-8")
print(abstract)

In [ ]:
# ============================================================
# 07. Create Methods draft
# ============================================================

methods = f'''
# Methods Draft

## Study design and data source
This study was designed as a reproducible machine learning benchmark using data from the Parkinson’s Progression Markers Initiative (PPMI). The workflow was organized into sequential notebooks covering data access verification, cohort definition, outcome construction, baseline predictor extraction, preprocessing, internal validation, multimodal feature integration, biomarker integration, and manuscript-ready reporting.

## Study population
The primary analytic cohort included participants with Parkinson’s disease who had sufficient baseline and follow-up MDS-UPDRS Part III data to define the motor progression outcome. The final analytic cohort contained {summary_values['n_total']} participants.

## Outcome definition
The primary outcome was rapid motor progression. Motor progression was defined using annualized change in MDS-UPDRS Part III from baseline to the selected follow-up visit. Participants in the upper quartile of annualized MDS-UPDRS Part III worsening were classified as rapid progressors. This produced {summary_values['n_rapid']} rapid progressors ({summary_values['pct_rapid']}%) in the analytic cohort.

## Predictor sets
Three predictor sets were evaluated:

1. **Clinical-only predictors:** baseline demographic, clinical, motor, non-motor, cognitive, medication-related, and related variables passing predefined missingness checks.
2. **Clinical + DaTSCAN/SBR predictors:** clinical predictors augmented with screening DaTSCAN quantitative SBR features.
3. **Clinical + DaTSCAN/SBR + SAA biomarkers:** multimodal predictors augmented with SAA-related biomarker features that passed missingness and overlap criteria.

## Preprocessing
All preprocessing was conducted after train/test splitting. Missing values were imputed using training data only. Continuous features were scaled where appropriate, and categorical variables were encoded using training-set mappings and then applied to the held-out test set. Features with excessive missingness or no variance were excluded before modeling.

## Model development and validation
Baseline and tuned models were evaluated using internal validation. Candidate algorithms included logistic regression variants, random forest models, gradient boosting models, and baseline dummy classifiers. Model selection was based on held-out test performance after cross-validation-based tuning within the training set.

## Performance metrics
Models were evaluated using ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, F1-score, and threshold-based summaries. Because rapid progressors represented approximately {summary_values['pct_rapid']}% of the cohort, PR-AUC, sensitivity, specificity, and balanced accuracy were emphasized in addition to ROC-AUC.

## Sensitivity analyses
Sensitivity analyses evaluated model performance after excluding baseline MDS-UPDRS Part III to assess whether models were overly dependent on baseline motor severity.
'''.strip()

(OUT_DIR / "03_methods_draft.md").write_text(methods, encoding="utf-8")
print(methods)

In [ ]:
# ============================================================
# 08. Create Results draft
# ============================================================

# Create compact performance table for text insertion
perf_cols = [c for c in ["analysis_stage", "feature_set", "model", "roc_auc", "pr_auc", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "threshold"] if c in performance.columns]
perf_for_md = performance[perf_cols].copy()

results = f'''
# Results Draft

## Cohort and outcome
The final analytic cohort included {summary_values['n_total']} participants with Parkinson’s disease. Rapid motor progression was identified in {summary_values['n_rapid']} participants ({summary_values['pct_rapid']}%). The mean baseline MDS-UPDRS Part III score was {summary_values['baseline_np3_mean']}, and the median baseline score was {summary_values['baseline_np3_median']}.

## Model performance
Overall model performance was modest. The best held-out ROC-AUC was {summary_values['best_roc_auc']} for {summary_values['best_roc_model']} using the {summary_values['best_roc_feature_set']} feature set. The best PR-AUC was {summary_values['best_pr_auc']}, and the best balanced accuracy was {summary_values['best_balanced_accuracy']}. These results indicate limited discrimination and reinforce that the models should not be presented as clinically deployable.

## Incremental value of multimodal features
Adding DaTSCAN/SBR features produced only small and inconsistent incremental gains compared with the clinical-only benchmark. SAA biomarker integration improved some sensitivity and F1-score trade-offs in selected models but did not consistently improve PR-AUC or specificity. Therefore, DaTSCAN/SBR should be presented as a secondary multimodal analysis, while SAA biomarker results should be interpreted as exploratory.

## Final model interpretation
The final model selection process favored a transparent benchmark interpretation. Clinical-only models are retained as the primary comparator. DaTSCAN/SBR models provide a secondary multimodal benchmark. SAA biomarker models are exploratory and hypothesis-generating. Across all analyses, the prediction of rapid motor progression remained challenging.

## Manuscript-ready performance table

{md_table_from_df(perf_for_md, max_rows=20)}
'''.strip()

(OUT_DIR / "04_results_draft.md").write_text(results, encoding="utf-8")
print(results[:3000])

In [ ]:
# ============================================================
# 09. Create table captions and figure legends
# ============================================================

captions = '''
# Table Captions and Figure Legends

## Table 1. Cohort and outcome characteristics
Summary of the final analytic cohort, including the number of Parkinson’s disease participants, rapid progressor frequency, and baseline motor severity.

## Table 2. Predictor feature sets
Summary of the clinical-only, clinical + DaTSCAN/SBR, and clinical + DaTSCAN/SBR + SAA biomarker feature sets used in the reproducible modeling workflow.

## Table 3. Held-out test performance of machine learning models
Comparison of model performance across feature sets using ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, and F1-score.

## Table 4. Incremental value of multimodal features
Summary of performance changes after adding DaTSCAN/SBR features and SAA biomarker features to the clinical baseline model.

## Table 5. Final model selection and scientific interpretation
Decision table describing the selected primary comparator, secondary multimodal analyses, exploratory biomarker analyses, and limitations regarding clinical deployment.

## Figure 1. Reproducible study workflow
Flow diagram summarizing the notebook-based workflow from data access verification through cohort definition, outcome construction, preprocessing, model development, multimodal integration, biomarker analysis, and manuscript-ready reporting.

## Figure 2. Outcome distribution
Distribution of annualized MDS-UPDRS Part III change and threshold used to define rapid motor progression.

## Figure 3. Model performance comparison
Bar plot or point-range plot comparing ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, and F1-score across feature sets.

## Figure 4. Incremental value of DaTSCAN/SBR and SAA biomarkers
Visualization of the performance difference between clinical-only, clinical + DaTSCAN/SBR, and clinical + DaTSCAN/SBR + SAA models.

## Figure 5. Feature importance of the selected exploratory model
Permutation importance or model-derived feature importance for the selected model, interpreted as exploratory rather than causal.
'''.strip()

(OUT_DIR / "05_table_captions_and_figure_legends.md").write_text(captions, encoding="utf-8")
print(captions)

In [ ]:
# ============================================================
# 10. Create limitations and future work
# ============================================================

limitations = '''
# Limitations and Future Work

## Limitations

1. **Internal validation only.** The current analysis used an internal train/test split. External validation in independent cohorts is required before clinical interpretation.
2. **Modest discrimination.** The best held-out ROC-AUC remained close to 0.60, indicating limited predictive accuracy.
3. **Class imbalance.** Rapid progressors represented approximately one quarter of the analytic cohort, which affects sensitivity, PR-AUC, and threshold selection.
4. **Outcome definition.** Rapid progression was defined using an upper-quartile threshold of annualized MDS-UPDRS Part III change. Alternative definitions may yield different results.
5. **Baseline severity sensitivity.** Baseline MDS-UPDRS Part III is clinically informative but may contribute to regression-to-the-mean effects; sensitivity analysis without baseline motor score should be reported.
6. **Multimodal missingness and availability.** Imaging and biomarker predictors were included only when sufficient coverage was available, limiting broader multimodal inference.
7. **Exploratory biomarker interpretation.** SAA-related variables were useful for exploratory analysis but did not consistently improve all performance metrics.
8. **No clinical deployment claim.** The models should not be presented as ready for clinical decision-making.

## Future Work

1. Validate the workflow in external cohorts such as AMP-PD non-PPMI cohorts when variable harmonization permits.
2. Evaluate alternative progression definitions, including continuous progression modeling and time-to-event formulations.
3. Explore longitudinal feature engineering using repeated measures before the prediction horizon.
4. Test more advanced multimodal methods only after establishing robust external validation.
5. Perform calibration and decision-curve analysis in independent datasets.
6. Develop a fully reproducible public code repository with synthetic/example data loaders and clear instructions for approved PPMI users.
'''.strip()

(OUT_DIR / "06_limitations_and_future_work.md").write_text(limitations, encoding="utf-8")
print(limitations)

In [ ]:
# ============================================================
# 11. Create data availability, ethics, and acknowledgement draft
# ============================================================

data_availability = '''
# Data Availability, Ethics, and Acknowledgement Draft

## Data availability
Data used in this study were obtained from the Parkinson’s Progression Markers Initiative (PPMI). Individual-level PPMI data are available to qualified researchers through the PPMI/LONI data access process subject to applicable data use agreements. Raw participant-level data should not be redistributed through public repositories by downstream users.

The analysis code, notebook workflow, environment description, and non-identifiable derived summary tables should be deposited in a public repository, subject to PPMI data use requirements. Because the PPMI database may evolve over time, the repository should document the exact data download date and file versions used in the analysis.

## Ethics statement
This secondary analysis used de-identified data from PPMI. PPMI obtained study approvals and participant consent according to its study protocols. The present analysis used existing controlled-access data and did not involve new participant recruitment or biospecimen collection.

## PPMI data source acknowledgement template
Data used in the preparation of this article were obtained on [YYYY-MM-DD] from the Parkinson’s Progression Markers Initiative (PPMI) database. For up-to-date information on the study, visit the PPMI website.

## Funding acknowledgement template
PPMI is a public-private partnership funded by The Michael J. Fox Foundation for Parkinson’s Research and its funding partners. The final manuscript should include the full current PPMI funding partner acknowledgement as required by PPMI publication policy.

## Publication compliance reminder
Before journal submission, verify the current PPMI publication policy and submit the manuscript or abstract for the required administrative review if applicable.
'''.strip()

(OUT_DIR / "07_data_availability_ethics_acknowledgement.md").write_text(data_availability, encoding="utf-8")
print(data_availability)

In [ ]:
# ============================================================
# 12. Create reporting checklist and submission readiness checklist
# ============================================================

reporting_items = [
    ("Title identifies ML benchmark and Parkinson’s progression", "YES", "Use benchmark wording; avoid clinical deployment claim."),
    ("Data source clearly identified as PPMI", "YES", "Include data download date and PPMI acknowledgement."),
    ("Cohort selection described", "YES", "Use Notebook 02 and Table 1 outputs."),
    ("Outcome definition described", "YES", "Annualized MDS-UPDRS Part III change; rapid progression upper quartile."),
    ("Predictor timing described", "YES", "Baseline/screening predictors only."),
    ("Data leakage precautions described", "YES", "Train-only preprocessing and no future predictors."),
    ("Train/test split described", "YES", "Stratified held-out test split from Notebook 04."),
    ("Missing data handling described", "YES", "Imputation fitted on training set only."),
    ("Performance metrics reported", "YES", "ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, F1."),
    ("Clinical deployment limitations stated", "YES", "Explicitly state models are not clinically deployable."),
    ("Sensitivity analysis without baseline_NP3TOT reported", "YES", "Include as supplementary table."),
    ("External validation limitation stated", "YES", "Required before clinical application."),
]

reporting_checklist = pd.DataFrame(reporting_items, columns=["item", "status", "notes"])
reporting_checklist.to_csv(OUT_DIR / "08_reporting_checklist.csv", index=False)

submission_items = [
    ("Confirm PPMI data download date", "TODO"),
    ("Insert current PPMI RRID/data source statement if required", "TODO"),
    ("Insert full current PPMI funding partner acknowledgement", "TODO"),
    ("Confirm journal formatting requirements", "TODO"),
    ("Create figures from approved outputs", "TODO"),
    ("Create supplementary notebook/code repository", "TODO"),
    ("Prepare README with reproducibility instructions", "TODO"),
    ("Do not upload raw PPMI participant data to public repository", "REQUIRED"),
    ("Run manuscript through PPMI administrative review before submission if required", "TODO"),
]

submission_checklist = pd.DataFrame(submission_items, columns=["item", "status"])
submission_checklist.to_csv(OUT_DIR / "09_submission_readiness_checklist.csv", index=False)

display(reporting_checklist)
display(submission_checklist)

In [ ]:
# ============================================================
# 13. Save final summary report
# ============================================================

output_files = sorted([p.name for p in OUT_DIR.glob("*")])

summary_report = f'''
Notebook 14 — Manuscript Drafting Package Summary

Status: COMPLETED

Input source:
- Notebook 13 manuscript-ready outputs were successfully located and loaded.

Key manuscript positioning:
- Reproducible machine learning benchmark.
- Not a clinically deployable prediction model.

Core results:
- Analytic cohort: {summary_values['n_total']}
- Rapid progressors: {summary_values['n_rapid']} ({summary_values['pct_rapid']}%)
- Best held-out ROC-AUC: {summary_values['best_roc_auc']}
- Best ROC-AUC model: {summary_values['best_roc_model']}
- Best ROC-AUC feature set: {summary_values['best_roc_feature_set']}
- Best PR-AUC: {summary_values['best_pr_auc']}
- Best balanced accuracy: {summary_values['best_balanced_accuracy']}
- Best F1-score: {summary_values['best_f1']}

Generated files:
''' + "\n".join([f"- {x}" for x in output_files]) + '''

Next recommended step:
- Notebook 15 or manuscript document creation: convert this drafting package into a full manuscript draft with references, tables, and figures.
'''

(OUT_DIR / "10_notebook_14_summary_report.txt").write_text(summary_report.strip(), encoding="utf-8")

print(summary_report)

## Scientific Interpretation

The approved results support a manuscript framed as a transparent, reproducible benchmark. The models show modest discrimination, and multimodal additions produced limited or inconsistent incremental value. Therefore, the manuscript should emphasize:

- reproducibility,
- transparent benchmarking,
- careful data leakage prevention,
- realistic reporting of limited predictive performance,
- and the need for external validation.

## Quality Control Checklist

The notebook generates:

- `08_reporting_checklist.csv`
- `09_submission_readiness_checklist.csv`
- `10_notebook_14_summary_report.txt`

Notebook 14 should be considered complete when all expected output files are created in:

`outputs/notebook_14_manuscript_drafting_package/`

## Expected Output

Expected files:

1. `01_title_and_key_messages.md`
2. `02_structured_abstract_draft.md`
3. `03_methods_draft.md`
4. `04_results_draft.md`
5. `05_table_captions_and_figure_legends.md`
6. `06_limitations_and_future_work.md`
7. `07_data_availability_ethics_acknowledgement.md`
8. `08_reporting_checklist.csv`
9. `09_submission_readiness_checklist.csv`
10. `10_notebook_14_summary_report.txt`